# Exploring the ISMIP6 Icechunk Store

This notebook explores the three store types (combined, state, flux) and demonstrates how to compare variables across models and experiments.

In [ ]:
import icechunk
import xarray as xr
import matplotlib.pyplot as plt
import zarr

## Connect to the store

In [ ]:
SOURCE_BUCKET = "s3://us-west-2.opendata.source.coop/englacial/ismip6"

storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix="englacial/ismip6/icechunk-ais",
    region="us-west-2",
    anonymous=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        SOURCE_BUCKET + "/",
        store=icechunk.s3_store(region="us-west-2", anonymous=True),
    )
)
credentials = icechunk.containers_credentials({SOURCE_BUCKET + "/": None})

repo = icechunk.Repository.open(
    storage=storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)
session = repo.readonly_session(branch="main")
root = zarr.open(session.store, mode='r')

## Explore the three store types

In [ ]:
# Top-level groups
for group_name in sorted(root.keys()):
    group = root[group_name]
    n_models = len(list(group.keys()))
    print(f"{group_name}/ ({n_models} models)")

In [ ]:
# List experiments for a model in each store type
model = "AWI_PISM1"
for store_type in ["combined", "state", "flux"]:
    try:
        exps = sorted(root[store_type][model].keys())
        print(f"{store_type}/{model}: {len(exps)} experiments")
        print(f"  {exps[:5]}{'...' if len(exps) > 5 else ''}")
    except KeyError:
        print(f"{store_type}/{model}: not found")

## Compare combined vs state/flux

The combined store merges all variables and bins time to annual resolution.
The state and flux stores keep native time resolution but only include their respective variable types.

In [ ]:
# Open the same model/experiment from each store type
model, exp = "AWI_PISM1", "exp05"

for store_type in ["combined", "state", "flux"]:
    group_path = f"{store_type}/{model}/{exp}"
    ds = xr.open_zarr(session.store, group=group_path, consolidated=False)
    print(f"\n{group_path}:")
    print(f"  Variables: {sorted(ds.data_vars)}")
    if 'time' in ds:
        print(f"  Time: {ds.time.values[0]} to {ds.time.values[-1]} ({len(ds.time)} steps)")

## Compare a variable across models

In [ ]:
# Compare ice thickness (lithk) across 4 models for exp05
models = ["AWI_PISM1", "DOE_MALI", "NCAR_CISM", "PIK_PISM1"]
exp = "exp05"

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, model in zip(axes.flat, models):
    ds = xr.open_zarr(session.store, group=f"combined/{model}/{exp}", consolidated=False)
    ds['lithk'].isel(time=0).plot(ax=ax, cmap='viridis', add_colorbar=True)
    ax.set_aspect('equal')
    ax.set_title(f"{model}")

fig.suptitle(f"Ice thickness (lithk) - {exp} - first timestep", fontsize=14)
plt.tight_layout()

## Check ignore_value annotations

The ingest pipeline annotates arrays with `ignore_value` attributes where sentinel values (e.g. 0.0 in ocean regions) were detected.

In [ ]:
# Check which variables have ignore_value annotations
group = root['combined/AWI_PISM1/exp05']
for var_name in sorted(group.keys()):
    arr = group[var_name]
    if 'ignore_value' in arr.attrs:
        attrs = arr.attrs
        print(f"  {var_name}: ignore_value={attrs['ignore_value']}", end="")
        if 'valid_min' in attrs:
            print(f", valid_range=[{attrs['valid_min']:.4g}, {attrs['valid_max']:.4g}]", end="")
        print()